# 02. Circuit Validation

This notebook demonstrates the **causal validation phase**.

Discovery tells us where to look. This notebook asks the stricter question on the
**96-pair within-family matched cohort** that now anchors the repo's active claim:
- If we remove or replace part of the model state, does the decision actually change?

In mechanistic terms, this notebook focuses on:
- **grouped path patching** for sufficiency
- **grouped head ablation** for necessity
- comparison between a minimal direct route and a broader late carrier

In plain language:
- What parts of the model are actually doing causal work?
- Which parts are just nearby or correlated?


In [ ]:
from pathlib import Path
import math
import textwrap

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(
    context="talk",
    style="whitegrid",
    palette="deep",
    rc={
        "figure.dpi": 120,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.facecolor": "#FCFCFC",
        "figure.facecolor": "white",
        "grid.color": "#D9DDE3",
        "grid.linewidth": 0.8,
        "axes.edgecolor": "#4A5568",
        "axes.labelcolor": "#1A202C",
        "xtick.color": "#2D3748",
        "ytick.color": "#2D3748",
        "axes.titleweight": "semibold",
    },
)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 140)


def find_project_root() -> Path:
    candidates = [Path.cwd()] + list(Path.cwd().parents)
    for candidate in candidates:
        if (candidate / "artifacts").exists() and (candidate / "scaled_validation.py").exists():
            return candidate
        if (candidate / "mech-interp-circuit" / "artifacts").exists():
            return candidate / "mech-interp-circuit"
    raise FileNotFoundError("Could not find mech-interp-circuit project root from the current working directory.")


PROJECT_ROOT = find_project_root()
ARTIFACTS = PROJECT_ROOT / "artifacts"
print("Project root:", PROJECT_ROOT)
print("Artifacts dir:", ARTIFACTS)


def read_csv(name: str) -> pd.DataFrame:
    path = ARTIFACTS / name
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def clip_text(text: str, *, width: int = 180) -> str:
    text = str(text).strip().replace("\r\n", "\n")
    text = " ".join(text.split())
    if len(text) <= width:
        return text
    return text[: width - 3] + "..."


def show_barh(df, label_col, value_col, *, title, xlabel, color="#2B6CB0", sort=True):
    plot_df = df.copy()
    if sort:
        plot_df = plot_df.sort_values(value_col, ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(plot_df))))
    sns.barplot(data=plot_df, x=value_col, y=label_col, ax=ax, color=color, orient="h")
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")
    sns.despine(ax=ax, left=False, bottom=False)
    plt.tight_layout()
    return fig, ax


## Step 1: Load the core validation artifacts

We use grouped late-route patching and ablation summaries on the 96-pair cohort.
Historical 18-pair pilot artifacts are intentionally excluded from the main validation narrative.


In [ ]:
route_minimal = read_csv("circuit_val_path_patching_h011_plus_l12_top3_combo96_h100_summary.csv")
route_top5 = read_csv("circuit_val_path_patching_l12_writer_top5_combo96_h100_summary.csv")
route_minus_h2 = read_csv("circuit_val_path_patching_l12_writer_minus_h2_combo96_h100_summary.csv")
route_minus_h28 = read_csv("circuit_val_path_patching_l12_writer_minus_h28_combo96_h100_summary.csv")
ablate_minimal = read_csv("circuit_val_head_group_ablation_l12_h011_route_combo96_h100_summary.csv")
ablate_top5 = read_csv("circuit_val_head_group_ablation_l12_writer_top5_combo96_h100_summary.csv")
ablate_minus_h2 = read_csv("circuit_val_head_group_ablation_l12_writer_minus_h2_combo96_h100_summary.csv")
ablate_h28 = read_csv("circuit_val_head_group_ablation_l12_writer_h28_combo96_h100_summary.csv")

route_minimal


## Step 2: Compare the candidate late routes by path patching

Path patching tests sufficiency: if we replace a candidate route with the benign version,
how much does the malicious-vs-benign decision move?


In [ ]:
route_rows = [
    ("Minimal direct route", float(route_minimal["mean_delta"].iloc[0]), float(route_minimal["flip_rate"].iloc[0])),
    ("Top-5 late bundle", float(route_top5["mean_delta"].iloc[0]), float(route_top5["flip_rate"].iloc[0])),
    ("Top-4 without H2", float(route_minus_h2["mean_delta"].iloc[0]), float(route_minus_h2["flip_rate"].iloc[0])),
    ("Top-4 without H28", float(route_minus_h28["mean_delta"].iloc[0]), float(route_minus_h28["flip_rate"].iloc[0])),
]
route_df = pd.DataFrame(route_rows, columns=["route", "mean_delta", "flip_rate"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(route_df["route"], route_df["mean_delta"], color="#2F855A")
axes[0].set_title("Path Patching on 96-Pair Cohort")
axes[0].set_ylabel("Mean logit delta")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(route_df["route"], route_df["flip_rate"], color="#2B6CB0")
axes[1].set_title("Flip Rate on 96-Pair Cohort")
axes[1].set_ylabel("Flip rate")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

route_df


## Step 2b: Visualize a validated upstream attention head

The grouped route result tells us which path matters, but it does not show what one of the important heads is actually attending to.

The optional cell below loads a real malicious prompt from the 96-pair cohort, runs a cached forward pass, and opens a `circuitsvis` attention-pattern viewer for `L0H11`, the cleanest validated upstream head in the writeup.

This is intentionally separate from the CSV-only workflow because it is heavier:
- it loads the model
- it computes a real attention cache
- it visualizes the full head-pattern tensor so the reader can inspect it interactively


In [ ]:
# Optional heavier cell: inspect a real attention pattern for the validated upstream head.
#
# import circuitsvis as cv
# import sys
#
# sys.path.append(str(PROJECT_ROOT))
# from scaled_validation import build_hooked_transformer, load_hf_model_and_tokenizer, make_prompt
#
# manifest = read_csv("circuit_val_pair_manifest_t3000_combo_cap20_valid_h100.csv")
# mal_row = (
#     manifest[(manifest["label"] == "malicious") & (manifest["pair_indicator"] == "Invoke-WebRequest")]
#     .sort_values("logit_diff", ascending=False)
#     .iloc[0]
# )
#
# BEST_LAYER, BEST_HEAD = 0, 11
# hf_model, tokenizer, device = load_hf_model_and_tokenizer(device="cpu", torch_dtype="float32")
# model = build_hooked_transformer(
#     hf_model,
#     tokenizer,
#     device=device,
#     torch_dtype="float32",
#     template_name="meta-llama/Llama-3.1-8B-Instruct",
#     first_n_layers=4,
#     use_attn_result=False,
# )
#
# prompt = make_prompt(mal_row["content"])
# mal_toks = model.to_tokens(prompt)
# _, mal_cache = model.run_with_cache(mal_toks, return_type="logits")
#
# pattern_key = f"blocks.{BEST_LAYER}.attn.hook_pattern"
# attn_pattern = mal_cache[pattern_key][0].detach().cpu().numpy()
# tok_strs = [model.to_string(t.unsqueeze(0)) for t in mal_toks[0]]
#
# cv.attention.attention_patterns(attention=attn_pattern, tokens=tok_strs)


## Step 3: Compare the same routes by grouped ablation

Grouped ablation tests necessity: if we zero a candidate late route, how much of the
malicious decision disappears?

This is also where the role of `H2` changes. On the 96-pair cohort, removing `H2`
improves patching, but including it strengthens grouped ablation.


In [ ]:
ablation_rows = [
    ("Minimal late route", float(ablate_minimal["mean_delta"].iloc[0]), float(ablate_minimal["flip_rate"].iloc[0])),
    ("Top-5 late bundle", float(ablate_top5["mean_delta"].iloc[0]), float(ablate_top5["flip_rate"].iloc[0])),
    ("Top-4 without H2", float(ablate_minus_h2["mean_delta"].iloc[0]), float(ablate_minus_h2["flip_rate"].iloc[0])),
    ("Top-4 without H28", float(ablate_h28["mean_delta"].iloc[0]), float(ablate_h28["flip_rate"].iloc[0])),
]
ablation_df = pd.DataFrame(ablation_rows, columns=["route", "mean_delta", "flip_rate"])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(ablation_df["route"], ablation_df["mean_delta"], color="#C53030")
axes[0].set_title("Grouped Ablation on 96-Pair Cohort")
axes[0].set_ylabel("Mean logit delta")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(ablation_df["route"], ablation_df["flip_rate"], color="#2B6CB0")
axes[1].set_title("Flip Rate After Grouped Ablation")
axes[1].set_ylabel("Flip rate")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

ablation_df


## Step 4: What the 96-pair validation phase shows

The cleanest final validation claim is:

- the cleanest minimal direct route is `L0H11 -> L12H15/L12H5/L12H4`
- the cleaner sufficiency-oriented late carrier is `L12H15/L12H5/L12H4/L12H28`
- `L12H2` behaves more like a family-sensitive helper than a stable core writer
- these are the claims the repo now treats as validated, because they are measured directly on the 96-pair cohort

This is why the repo writeup separates:
- a **minimal direct branch** for mechanistic clarity
- a **broader late carrier** for stronger sufficiency on the larger cohort


## Optional: Rerun notes

The commands below are examples only. They are usually more comfortable on GPU, but they document exactly how the validation artifacts were produced.


In [ ]:
# Example only. Uncomment to rerun a small validation command.
#
# !python ../scaled_validation.py batch-head-group-ablation \
#     --manifest ../artifacts/circuit_val_pair_manifest_t3000_combo_cap20_valid_h100.csv \
#     --heads 12.15,12.5,12.4,12.28 \
#     --device cpu \
#     --torch-dtype float32 \
#     --num-pairs 2 \
#     --allow-zero-indicator-malicious \
#     --output-prefix ../artifacts/demo_validation_cpu
